# 04. GARCH(1,1) 변동성 예측

## 목적
매월 말 기준, 다음 달 개별 종목 월별 실현변동성(`vol_21d`) 예측

## 설계
- 모델: **GARCH(1,1)** — 03 EDA에서 ARCH 효과 85% 유의 확인
- 입력: 직전 60개월 월별 수익률
- 타겟: 다음 달 `vol_21d` (연환산)
- 검증: Rank IC (Spearman) — 03의 AR(1) R² 중앙값 0.235와 비교
- 출력: `data/vol_predicted.csv` — (date, ticker, vol_pred)

In [2]:
import pandas as pd
import numpy as np
from arch import arch_model
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import warnings, time, platform
from pathlib import Path

warnings.filterwarnings('ignore')
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'data'
OUT_DIR  = BASE_DIR / 'outputs' / '04_garch'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_WINDOW = 60
MIN_OBS      = 36
START_PRED   = '2011-01-01'
ANN          = np.sqrt(12)

panel = pd.read_csv(DATA_DIR / 'monthly_panel.csv', parse_dates=['date'])
panel = panel.set_index(['date', 'ticker'])
ret_pivot = panel['ret_1m'].unstack('ticker')
vol_pivot = panel['vol_21d'].unstack('ticker')

all_dates  = ret_pivot.index.sort_values()
pred_dates = all_dates[all_dates >= START_PRED]

print(f'패널: {panel.shape}')
print(f'예측 기간: {pred_dates[0].date()} ~ {pred_dates[-1].date()} ({len(pred_dates)}개월)')

패널: (100184, 12)
예측 기간: 2011-01-31 ~ 2025-12-31 (180개월)


# GARCH(1,1) 예측 함수

In [3]:
def predict_garch(r, min_obs=MIN_OBS):
    """
    GARCH(1,1) 1-step ahead 연환산 변동성 예측
    수렴 실패 또는 데이터 부족 시 역사적 변동성으로 대체
    """
    r = r.dropna() * 100  # percent 스케일 (수치 안정성)
    if len(r) < min_obs:
        return float(r.std() * ANN / 100)
    try:
        am  = arch_model(r, vol='Garch', p=1, q=1, dist='normal', rescale=False)
        res = am.fit(disp='off', show_warning=False)
        fc  = res.forecast(horizon=1, reindex=False)
        return float(np.sqrt(fc.variance.values[0, 0]) * ANN / 100)
    except Exception:
        return float(r.std() * ANN / 100)

print('GARCH(1,1) 함수 정의 완료')
print(f'예상 소요 시간: 30~90분 (CPU 환경에 따라 다름)')

GARCH(1,1) 함수 정의 완료
예상 소요 시간: 30~90분 (CPU 환경에 따라 다름)


# Walk-forward 예측 루프

In [4]:
t0 = time.time()
all_records = []
n_fallback  = 0

print(f'GARCH(1,1) Walk-forward: {pred_dates[0].date()} ~ {pred_dates[-1].date()}')

for i, pred_date in enumerate(pred_dates):
    idx         = all_dates.get_loc(pred_date)
    train_dates = all_dates[max(0, idx - TRAIN_WINDOW): idx]
    train_ret   = ret_pivot.reindex(index=train_dates)

    if pred_date not in vol_pivot.index:
        continue
    universe = vol_pivot.loc[pred_date].dropna().index.tolist()

    for ticker in universe:
        if ticker in train_ret.columns:
            vol_pred = predict_garch(train_ret[ticker])
        else:
            vol_pred = np.nan
            n_fallback += 1
        all_records.append({'date': pred_date, 'ticker': ticker, 'vol_pred': vol_pred})

    if (i + 1) % 12 == 0:
        elapsed = time.time() - t0
        eta     = elapsed / (i + 1) * (len(pred_dates) - i - 1)
        print(f'  {pred_date.date()} ({i+1}/{len(pred_dates)}) '
              f'— 경과 {elapsed/60:.1f}분, 잔여 {eta/60:.1f}분')

print(f'\n완료: {(time.time()-t0)/60:.1f}분  |  예측 {len(all_records):,}개  |  fallback {n_fallback}개')

GARCH(1,1) Walk-forward: 2011-01-31 ~ 2025-12-31
  2011-12-31 (12/180) — 경과 1.0분, 잔여 14.7분
  2012-12-31 (24/180) — 경과 2.1분, 잔여 13.8분
  2013-12-31 (36/180) — 경과 3.2분, 잔여 12.8분
  2014-12-31 (48/180) — 경과 4.3분, 잔여 11.7분


KeyboardInterrupt: 

# Rank IC (참고용)\n",
    "\n",
    "어느 방법이 더 나은지는 포트폴리오 성과로 판단한다.  \n",
    "Rank IC는 예측 정확도 참고용으로만 확인한다.

In [ ]:
# ── 저장 ─────────────────────────────────────────────────────
vol_pred_df = pd.DataFrame(all_records)
vol_pred_df['date'] = pd.to_datetime(vol_pred_df['date'])
vol_pred_df.to_csv(DATA_DIR / 'vol_predicted.csv', index=False)
print(f'저장: {DATA_DIR}/vol_predicted.csv  ({len(vol_pred_df):,}행)')

# ── Rank IC 평가 ─────────────────────────────────────────────
vol_pred_pivot = vol_pred_df.set_index(['date', 'ticker'])['vol_pred'].unstack()
ic_records = []

for pred_date in pred_dates:
    if pred_date not in vol_pred_pivot.index or pred_date not in vol_pivot.index:
        continue
    pred   = vol_pred_pivot.loc[pred_date].dropna()
    actual = vol_pivot.loc[pred_date].dropna()
    common = pred.index.intersection(actual.index)
    if len(common) < 30:
        continue
    ic, _ = spearmanr(pred[common], actual[common])
    ic_records.append({'date': pred_date, 'ic': ic})

ic_df = pd.DataFrame(ic_records).set_index('date')

print('\n' + '='*50)
print('GARCH(1,1) Rank IC (예측 vol vs 실제 vol_21d)')
print('='*50)
print(f'  평균 IC:    {ic_df["ic"].mean():.4f}')
print(f'  중앙값 IC:  {ic_df["ic"].median():.4f}')
print(f'  양수 비율:  {(ic_df["ic"] > 0).mean():.1%}')
print(f'  ICIR:       {ic_df["ic"].mean() / ic_df["ic"].std():.4f}')
print(f'\n  [비교] AR(1) baseline R² 중앙값: 0.235 (03 결과)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('GARCH(1,1) 변동성 예측 성능', fontsize=13, fontweight='bold')

ax = axes[0]
ic_df['ic'].plot(ax=ax, color='#1976D2', alpha=0.5, linewidth=1)
ic_df['ic'].rolling(12).mean().plot(ax=ax, color='#E53935', linewidth=2,
                                     label=f'12M 이동평균')
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(ic_df['ic'].mean(), color='green', linewidth=1, linestyle='--',
           label=f'평균 IC = {ic_df["ic"].mean():.3f}')
ax.set_title('월별 Rank IC')
ax.set_ylabel('Spearman IC')
ax.legend(fontsize=9)

ax = axes[1]
ax.hist(ic_df['ic'], bins=30, color='#1976D2', alpha=0.8, edgecolor='white')
ax.axvline(ic_df['ic'].mean(), color='red', linestyle='--',
           label=f'평균 {ic_df["ic"].mean():.3f}')
ax.axvline(0, color='black', linewidth=1)
ax.set_title('IC 분포')
ax.set_xlabel('Rank IC')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / 'garch_rank_ic.png', dpi=150, bbox_inches='tight')
plt.show()

# 예측값 저장
05_BlackLitterman에서 사용할 예측 변동성 저장.

In [ ]:
# 두 예측값 저장 — 05에서 각각 BL 구성, 06에서 성과 비교
vol_pred = comparison[['vol_baseline', 'vol_ridge']].copy()
vol_pred.to_csv(DATA_DIR / 'vol_predicted.csv')

print(f'저장 완료: {DATA_DIR / "vol_predicted.csv"}')
print(f'형태: {vol_pred.shape}')
print(f'기간: {vol_pred.index.get_level_values("date").min().date()} ~ '
      f'{vol_pred.index.get_level_values("date").max().date()}')
print()
print('Rank IC (참고)')
print(f'  Baseline: {ic_base_df["rank_ic"].mean():.3f}')
print(f'  Ridge:    {ic_ridge_df["rank_ic"].mean():.3f}')
print()
print('→ 어느 방법이 나은지는 06_Comparison에서 포트폴리오 성과로 판단')